# Previsão de volume do Sistema Cantareira (CT_TOTAL) via LSTM — grade de experimentos para o artigo (H = 3 e H = 12)

Este notebook consolida os **resultados de previsão supervisionada do
CT_TOTAL para o artigo do pós-doutorado**. Ele refaz, em um único
protocolo, a grade de experimentos validada em
`Notebooks/LSTM/lstm_exploratory.ipynb` (Parte II), agora nos **dois horizontes** da
pesquisa:

| | base **persistência** ($\hat V = \text{Vol}(t) + \hat\Delta$) | base **climatologia** ($\hat V = \text{Clim}[m] + \hat\Delta$) |
|---|---|---|
| **A** — mínimo | `A` | `A*` |
| **B** — + operacionais (roteamento completo) | `B` | `B*` |
| **C** — + balanço operacional (`qbal`) | `C` | `C*` |
| **D** — + `qbal` + `SRI12` | `D` | `D*` |

São **8 experimentos por horizonte** (A/B/C/D × duas bases) × **2
horizontes (H = 3 e H = 12 meses)** = **16 experimentos**, todos com:

* janela de entrada **W = 12 meses** (`seq_len = 12`);
* **walk-forward expansivo** com 7 folds anuais (2019–2025) e janela de
  validação de 12 meses (protocolo herdado do `lstm_exploratory` (Parte II), §5);
* **10 seeds por fold** (predição = média do ensemble entre seeds);
* comparação obrigatória com três baselines ingênuos (persistência-H,
  climatologia mensal expansiva, média do treino — `docs/methodological_rules.md` §3.5).

Na sequência: (§7) séries no período de teste para todos os modelos;
(§8) tabela de performance com as métricas do `lstm_exploratory` (Parte II)
(RMSE/MAE/r/NSE, decomposição KGE, POD/FAR/CSI/HSS, AUC) **mais os
principais coeficientes do treinamento** (RMSE/NSE de treino, RMSE de
validação, época ótima); (§9) seleção do melhor experimento por
horizonte via ranking multi-métrica; (§10) **regressão quantílica**
(P10/P50/P90) dos dois vencedores, com as bandas para o artigo; (§11)
**interpretabilidade** (saliência temporal, sensibilidade à perturbação
e Seq-PDP, análogas às do `lstm_exploratory.ipynb` (Parte I) §15).

**Notas de escopo e convenções**

* Notebook **estável**, adicionado a pedido da autora como entregável
  do artigo — espelha o desvio documentado do `lstm_exploratory` (Parte II) em relação
  à diretriz "um notebook por frente" do `docs/methodological_rules.md`.
* **Sem `tws`** em nenhum experimento (`docs/methodological_rules.md` §3.2: a série GRACE
  começa em 2002 e cortaria 18 anos de treino).
* `split_on="target_date"` **sempre** (`docs/methodological_rules.md` §3.1).
* Dados: série Xavier/BR-DWGD corrigida e promovida no M8
  (1984-01 → 2025-12). Não usa ERA5-Land.
* Saídas em `Previsao/Outputs/LSTM/` com prefixo/pasta `article` —
  não sobrescrevem os artefatos do `lstm_exploratory` (Parte II)/`lstm_exploratory` (Parte I).

## § 1. Delineamento experimental

### 1.1. As duas bases de reconstrução — alvos DIFERENTES

Todos os experimentos preveem uma **variação** ($\Delta$) e reconstroem
o volume somando-a a uma base; as duas bases definem **alvos de treino
diferentes** (não basta trocar a base na reconstrução — as previsões
ficariam sistematicamente enviesadas):

| experimento | alvo treinado | reconstrução |
|---|---|---|
| **A/B/C** (base persistência) | $\Delta = \text{Vol}(t{+}H) - \text{Vol}(t)$ | $\text{Vol}(t) + \hat\Delta$ |
| **A\*/B\*/C\*** (base climatologia) | $\Delta = \text{Vol}(t{+}H) - \text{Clim}[m(t{+}H)]$ | $\text{Clim}[m(t{+}H)] + \hat\Delta$ |

**Implementação sem alterar scripts** (idêntica ao `lstm_exploratory` (Parte II)
§1.3): prever $\text{Vol}(t{+}H) - \text{Clim}[m(t{+}H)]$ equivale a
prever o *nível* da série de **anomalias**
$\text{VolAnom}(t) = \text{Vol}(t) - \text{Clim}[m(t)]$. Os
experimentos `*` rodam `run_lstm_multi_seed(..., target_mode="level")`
sobre a série de anomalias, com a climatologia do mês-alvo somada de
volta na reconstrução. Efeito colateral documentado: nesses
experimentos a feature `Vol_t` é a **anomalia** do mês corrente
(informação equivalente ao nível, dado o calendário nas features).

### 1.2. O papel do horizonte

`build_sequence_supervised` trata `horizon` como parâmetro livre — a
regressão é **multi-step direta** (a janela observada até $t$ mapeia
diretamente o alvo em $t+H$, sem recursão). Duas particularidades:

* **H = 12**: $m(t{+}12) = m(t)$ ⇒ a persistência-12 já é sazonalmente
  alinhada, e a persistência da anomalia reconstrói **exatamente**
  $\text{Vol}(t)$ (verificação de consistência na §6).
* **H = 3**: $m(t{+}3) \neq m(t)$ ⇒ a identidade acima **não vale**; o
  `y_base` interno dos experimentos `*` vira uma "persistência
  sazonalmente ajustada" ($\text{Vol}(t) - \text{Clim}[m(t)] +
  \text{Clim}[m(t{+}3)]$). Por isso, **todas** as métricas de
  persistência reportadas usam a persistência verdadeira
  $\text{Vol}(t)$, tomada do experimento A.

### 1.3. Correspondência com o `lstm_exploratory` (Parte II)

A nomenclatura do artigo (A/B/C e A\*/B\*/C\*) mapeia 1:1 nos ids
internos daquele notebook: `A ≡ A_pers`, `A* ≡ A_clim`, etc. Os ids
internos são mantidos no código e nos CSVs exportados; rótulos com
asterisco aparecem em tabelas e figuras.

## § 2. Ambiente e imports

Reusa integralmente o pipeline de carregamento/agregação do braço
XGBoost (`Previsao.Scripts`) e o subpacote LSTM
(`Previsao.Scripts.LSTM`) — nenhum código de modelagem novo. A ordem
dos imports (numpy/pandas/matplotlib **antes** de torch) evita o
conflito de runtime OpenMP (OMP #15) do Anaconda/Windows.

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch  # após numpy/pandas — evita OMP #15 (duplo libiomp5md) no Anaconda/Windows

# Fixar 1 thread reduz (mas não elimina) a variação numérica entre
# execuções de treino em CPU — ver nota de reprodutibilidade na §8.
torch.set_num_threads(1)

# Sobe até a raiz do repositório (a pasta que contém 'Previsao/'),
# robusto a qualquer cwd e ao nome do diretório do clone.
NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents)
     if (p / 'Previsao').is_dir()),
    NOTEBOOK_DIR,
)
PREVISAO_DIR = REPO_ROOT / 'Previsao'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Previsao.Scripts import (
    FeatureConstructor,
    default_cantareira_total_spec,
    build_total_cantareira_inputs,
    set_paper_plot_style,
    save_figure,
    diebold_mariano_test,
)
from Previsao.Scripts.LSTM import (
    run_lstm_multi_seed,
    run_lstm_quantile_multi_seed,
    compute_regression_metrics,
    coverage_rate,
    mean_interval_width,
    winkler_score,
    compute_temporal_saliency,
    compute_perturbation_sensitivity,
    compute_seq_pdp,
    plot_xai_heatmap,
)

DATA_DIR = PREVISAO_DIR / "Dados"
OUT_DIR = PREVISAO_DIR / "Outputs" / "LSTM"
FIG_DIR = OUT_DIR / "figures" / "article"
XAI_DIR = FIG_DIR / "xai"
PRED_DIR = OUT_DIR / "predictions"
METRICS_DIR = OUT_DIR / "metrics"

for p in [OUT_DIR, FIG_DIR, XAI_DIR, PRED_DIR, METRICS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

set_paper_plot_style()
print("Ambiente OK.")
print(f"  REPO_ROOT = {REPO_ROOT}")
print(f"  DATA_DIR  = {DATA_DIR}")
print(f"  FIG_DIR   = {FIG_DIR}")

## § 3. Dados e agregação do Sistema Cantareira (CT_TOTAL)

Séries `wide` (`date` + uma coluna por bacia) agregadas em CT_TOTAL via
`build_total_cantareira_inputs` — soma para vazões/armazenamento, média
ponderada por área para Pr/ETo (pesos QGIS em
`cantareira_equivalent.CANTAREIRA_AREA_KM2`, `docs/methodological_rules.md` §3.7).
Cobertura 1984-01 → 2025-12 (Pr/ETo zonais promovidas no M8).

A `FeatureConstructor` fornece as colunas cruas + operacionais (set B)
+ índices de memória longa (set C). A LSTM **não usa lags explícitos**
— a janela W = 12 já carrega a memória de curto prazo.

In [ ]:
def std(relative_path: str) -> pd.DataFrame:
    # Carrega um CSV wide de Previsao/Dados/, padroniza e ordena por data.
    df = pd.read_csv(DATA_DIR / relative_path)
    df["date"] = pd.to_datetime(df["date"])
    return df.sort_values("date").reset_index(drop=True)


pr = std("Pr/Pr_monthly.csv")
eto = std("ETo/ETo_monthly.csv")
qnat = std("Q/Qnat2_monthly.csv")
vol = std("Vol/Vol_monthly.csv")
qbal = std("Q/Qbal_monthly.csv")
qout = std("Q/Qout_monthly.csv")
qt5 = std("Q/Qt5_monthly.csv")
qt6 = std("Q/Qt6_monthly.csv")
qt7 = std("Q/Qt7_monthly.csv")
qpbs = std("Q/Qpbs_net_monthly.csv")
qelev = std("Q/Qelev_monthly.csv")
tws = std("TWS/TWS_monthly.csv")   # carregado p/ o construtor; NÃO vira feature

# Agregação dos 4 reservatórios (JAGJAC+CACH+ATIB+PCASTR) em CT_TOTAL.
eq_total = build_total_cantareira_inputs(
    pr=pr, eto=eto, qnat=qnat, vol=vol,
    qbal=qbal, qout=qout, tws=tws,
    spec=default_cantareira_total_spec(),
)

fc_total = FeatureConstructor(
    basins=["CT_TOTAL"],
    lags=[1], roll_windows=[3, 6, 12], index_scales=[3, 12], stats_ops=["Mean"],
)
_, datasets_total = fc_total.build(
    pr=eq_total["pr"], eto=eq_total["eto"], qnat=eq_total["qnat"],
    qbal=eq_total["qbal"], qout=eq_total["qout"], tws=eq_total["tws"],
    qt5=qt5, qt6=qt6, qt7=qt7, qpbs_net=qpbs, qelev=qelev,
)


def add_calendar_features(datasets: dict) -> dict:
    # month_sin/month_cos/is_dry — o XGBoostModel faz isso internamente;
    # como a LSTM não passa por ele, replicamos aqui (idêntico ao
    # lstm_exploratory, Partes I e II).
    out = {}
    for basin, df in datasets.items():
        df = df.copy()
        df["date"] = pd.to_datetime(df["date"])
        m = df["date"].dt.month
        df["month_sin"] = np.sin(2 * np.pi * m / 12)
        df["month_cos"] = np.cos(2 * np.pi * m / 12)
        df["is_dry"] = m.isin([4, 5, 6, 7, 8, 9]).astype(int)
        out[basin] = df
    return out


datasets_total = add_calendar_features(datasets_total)

vol_ct_total = eq_total["vol"][["date", "CT_TOTAL"]].copy()
vol_ct_total["date"] = pd.to_datetime(vol_ct_total["date"])

print(f"CT_TOTAL -> {datasets_total['CT_TOTAL'].shape}, "
      f"{datasets_total['CT_TOTAL']['date'].min().date()} a "
      f"{datasets_total['CT_TOTAL']['date'].max().date()}")

## § 4. Feature sets e a grade de experimentos

Os feature sets seguem o braço XGBoost adaptados à LSTM:

* **A — mínimo**: hidroclimáticas cruas + volume + calendário.
* **B — operacionais**: A + roteamento operacional *completo* do CT_TOTAL
  (`qout`, `qbal` locais; `qpbs_net`, `qelev` externas — túneis internos se
  anulam no agregado).
* **C — A + `qbal`**: núcleo A + o balanço operacional agregado do sistema
  (uma única variável operacional).
* **D — A + `qbal` + `SRI12`**: C + o índice padronizado de vazão de 12 meses
  (memória hidrológica longa).

São **4 feature sets × 2 bases** (persistência/climatologia) × **2 horizontes**
= **16 experimentos**, hiperparâmetros idênticos. **Reuso incremental** nesta
rodada: A/B/C foram reaproveitados do run anterior (idênticos em definição) e
**apenas D foi treinado** — o `run_grid` detecta os CSVs do run prévio e treina
só o feature set novo, poupando ~⅔ do tempo.

In [ ]:
MIN_DATE = "1984-01-01"
HORIZONS = (3, 12)

_CAL = ["month_sin", "month_cos", "is_dry"]
FEATURE_SETS = {
    "A": ["pr", "eto", "q", "Vol_t"] + _CAL,
    "B": ["pr", "eto", "q", "Vol_t",
          "qout", "qbal", "qpbs_net", "qelev"] + _CAL,
    "C": ["pr", "eto", "q", "Vol_t", "qbal"] + _CAL,
    "D": ["pr", "eto", "q", "Vol_t", "qbal", "SRI12"] + _CAL,
}
BASES = ("pers", "clim")
EXP_IDS = [f"{fs}_{b}" for fs in FEATURE_SETS for b in BASES]

# Rótulos do artigo: base persistência = A/B/C; base climatologia = A*/B*/C*.
EXP_LABEL = {f"{fs}_{b}": (fs if b == "pers" else f"{fs}*")
             for fs in FEATURE_SETS for b in BASES}
LABEL2ID = {v: k for k, v in EXP_LABEL.items()}

MODEL_HP = {"hidden_size": 16, "num_layers": 1, "dropout": 0.0, "head_hidden": 8}
TRAIN_HP = {"lr": 1e-3, "weight_decay": 1e-4, "batch_size": 16,
            "max_epochs": 300, "patience": 30, "grad_clip": 1.0}

for fs, cols in FEATURE_SETS.items():
    assert "tws" not in cols
    print(f"  set {fs}: p={len(cols)}  {cols}")
print(f"\nExperimentos por horizonte ({len(EXP_IDS)}): "
      f"{[EXP_LABEL[e] for e in EXP_IDS]}")
print(f"Horizontes: {HORIZONS}  ->  {len(HORIZONS) * len(EXP_IDS)} experimentos")

## § 5. Protocolo de avaliação: walk-forward expansivo (7 folds, 10 seeds)

Protocolo herdado sem alteração do `lstm_exploratory` (Parte II) (§5), agora
parametrizado pelo horizonte. Para o fold com ano-teste $Y$:

* **teste**: `target_date` em $[Y\text{-}01, Y\text{-}12]$ (12 meses);
* **validação**: os 12 meses de `target_date` anteriores
  (`VAL_WINDOW_MONTHS = 12`) — early stopping. A janela de 12 foi
  adotada por análise de sensibilidade (empata em RMSE com 36 e é a
  única que produz detecção de secas severas; sobreviveu a 10 seeds);
* **treino**: de `MIN_DATE` até o início da validação — expansivo;
* `max_date` filtra a data de **EMISSÃO** e `target = issue + H`, logo
  `max_date = Y-12 − H meses` (para H = 12 recai no `max_date =
  val_end` do `lstm_exploratory` (Parte II); bug histórico documentado lá, §13.2).

**Recalculado em CADA fold** (evita vazamento): climatologia mensal
(base dos `*` e baseline), média do treino e limiares de seca (§8).

**Custo** (grade completa): 2 horizontes × 8 experimentos × 7 folds × 10 seeds = **1120
treinamentos** + QR (§10: 2 × 7 × 10 = 140) = **1260 treinamentos** —
ordem de 3–5 h em CPU. Checkpoints por fold em
`Outputs/LSTM/metrics/_checkpoint_article_*.csv`.

In [ ]:
VAL_WINDOW_MONTHS = 12                     # protocolo — ver §5
TEST_YEARS = list(range(2019, 2026))       # 7 folds: 2019..2025
SEEDS_WF = [42, 7, 123, 2024, 99, 1, 2, 3, 4, 5]   # 10 seeds

WF_KW = dict(
    basin="CT_TOTAL", seq_len=12, min_date=MIN_DATE,
    split_on="target_date", model_kind="lstm",
    **MODEL_HP, **TRAIN_HP, seeds=SEEDS_WF, verbose=False,
)


def fold_bounds(
    test_year: int, horizon: int,
) -> tuple[pd.Timestamp, pd.Timestamp, pd.Timestamp]:
    # max_date filtra a ISSUE date; target = issue + H. Para o teste
    # terminar em Y-12, max_date = Y-12 - H meses (ver §5).
    test_start = pd.Timestamp(f"{test_year}-01-01")
    test_end = pd.Timestamp(f"{test_year}-12-01")
    val_end = test_start - pd.DateOffset(months=1)
    train_end = val_end - pd.DateOffset(months=VAL_WINDOW_MONTHS)
    max_date = test_end - pd.DateOffset(months=horizon)
    return train_end, val_end, max_date


def clim_for_fold(train_end: pd.Timestamp) -> pd.Series:
    # Climatologia mensal usando SOMENTE o treino deste fold (expansiva).
    tw = (vol_ct_total["date"] >= MIN_DATE) & (vol_ct_total["date"] <= train_end)
    return (vol_ct_total.loc[tw]
            .assign(month=lambda d: d["date"].dt.month)
            .groupby("month")["CT_TOTAL"].mean())


def anom_series(clim: pd.Series) -> pd.DataFrame:
    out = vol_ct_total.copy()
    out["CT_TOTAL"] = out["CT_TOTAL"] - out["date"].dt.month.map(clim)
    return out


print(f"{len(TEST_YEARS)} folds: anos-teste {TEST_YEARS}")
for h in HORIZONS:
    for y in [TEST_YEARS[0], TEST_YEARS[-1]]:
        te, ve, md_ = fold_bounds(y, h)
        print(f"  H={h:>2} fold {y}: treino até {te.date()} | val até "
              f"{ve.date()} | último issue {md_.date()}")

## § 6. Execução da grade (2 horizontes × 8 experimentos × 7 folds × 10 seeds; reuso incremental treina só o feature set novo)

Para cada fold: recalcula climatologia/anomalia, treina os 8
experimentos, reconstrói os `*` para o espaço físico (com verificação
de consistência) e calcula os baselines expansivos. Além das predições
de teste, guardamos:

* **métricas de treino e validação por fold** (predição ensemble —
  média entre seeds) — alimentam os "coeficientes do treinamento" da
  tabela da §8;
* **época ótima e nº de épocas treinadas** (média entre seeds);
* os **artefatos da seed 42** de cada (experimento × fold) — modelo,
  janelas de teste escaladas e scaler — para a interpretabilidade (§11).

In [ ]:
def reconstruct_level(preds_anom: pd.DataFrame, clim: pd.Series) -> pd.DataFrame:
    # Soma Clim[m(t+H)] a y_pred/y_true/y_base (todas as linhas/splits).
    out = preds_anom.copy()
    out["target_date"] = pd.to_datetime(out["target_date"])
    out["issue_date"] = pd.to_datetime(out["issue_date"])
    ct = out["target_date"].dt.month.map(clim)
    for col in ["y_pred", "y_true", "y_base"]:
        out[col] = preds_anom[col].to_numpy() + ct.to_numpy()
    return out


def split_ensemble(preds_df: pd.DataFrame, split: str) -> pd.DataFrame:
    # Média/desvio entre seeds por target_date, para um split.
    d = preds_df[preds_df["split"] == split].copy()
    d["target_date"] = pd.to_datetime(d["target_date"])
    return (
        d.groupby("target_date")
        .agg(y_true=("y_true", "first"), y_pred=("y_pred", "mean"),
             y_pred_std=("y_pred", "std"), y_base=("y_base", "first"))
        .sort_index().reset_index()
    )


def _keep_fitted(fitted: dict) -> dict:
    # Guarda só o necessário para o xAI (§11): modelo, janelas de teste
    # escaladas, nomes de features e o scaler do fold (seed 42).
    return {"model": fitted["model"].eval(), "X_test": fitted["X_test"],
            "feature_cols": fitted["feature_cols"], "scaler": fitted["scaler"]}


def _collect(e: str, test_year: int, res_df: pd.DataFrame,
             preds_lvl: pd.DataFrame, fit_list: list,
             aggs: dict, rows_split: list, rows_info: list,
             fitted_store: dict) -> None:
    # Consolida um experimento×fold: ensemble de teste + métricas de
    # treino/val + info de treinamento + artefatos da seed 42.
    a = split_ensemble(preds_lvl, "test")
    assert len(a) == 12, f"{e}/{test_year}: n_test={len(a)} (esperado 12)"
    aggs[e] = a
    for sp in ("train", "val"):
        ens = split_ensemble(preds_lvl, sp)
        m = compute_regression_metrics(
            ens["y_true"].to_numpy(), ens["y_pred"].to_numpy())
        rows_split.append({
            "fold": test_year, "exp": e, "split": sp, "n": len(ens),
            "rmse": m["rmse"], "mae": m["mae"],
            "corr": m["corr"], "nse": m["nse"],
        })
    rows_info.append({
        "fold": test_year, "exp": e,
        "best_epoch": float(res_df["best_epoch"].mean()),
        "n_epochs": float(res_df["n_epochs_trained"].mean()),
    })
    fitted_store[e][test_year] = _keep_fitted(fit_list[0])


def _mean_train_by_fold(horizon: int) -> dict:
    return {y: float(vol_ct_total.loc[
        (vol_ct_total["date"] >= MIN_DATE)
        & (vol_ct_total["date"] <= fold_bounds(y, horizon)[0]),
        "CT_TOTAL"].mean()) for y in TEST_YEARS}


def _by_fold_rows(wf: pd.DataFrame, exp_ids, horizon: int,
                  baselines: bool) -> list:
    # Métricas de teste por fold, a partir das colunas pooled de wf_all.
    rows = []
    for ty in TEST_YEARS:
        w = wf[wf["fold"] == ty]
        y = w["y_true"].to_numpy()
        preds = {EXP_LABEL[e]: w[f"y_{e}"].to_numpy() for e in exp_ids}
        if baselines:
            preds[f"persistência-{horizon}"] = w["y_base_persist"].to_numpy()
            preds["média treino"] = w["y_base_mean_train"].to_numpy()
            preds["climatologia"] = w["y_base_clim"].to_numpy()
        for lab, yh in preds.items():
            m = compute_regression_metrics(y, yh)
            rows.append({"fold": ty, "modelo": lab, "n_test": len(y),
                         "rmse": m["rmse"], "mae": m["mae"],
                         "corr": m["corr"], "nse": m["nse"]})
    return rows


def run_grid(horizon: int, train_only=None) -> dict:
    """Grade walk-forward para um horizonte.

    ``train_only=None`` treina TODOS os feature sets (auto-suficiente).
    ``train_only={"D"}`` treina só D e **reaproveita** A/B/C do run anterior
    (``article_preds_wf_h{h}.csv`` + ``article_*_by_fold``), evitando
    re-treinar. Os CSVs reusados devem corresponder às definições atuais de
    A/B/C (aqui: A, B e C=A+qbal, idênticos ao run imediatamente anterior).
    """
    t0 = time.time()
    all_fs = list(FEATURE_SETS)
    train_fs = all_fs if train_only is None else [f for f in all_fs if f in train_only]
    reuse = train_only is not None and len(train_fs) < len(all_fs)
    trained_ids = [f"{fs}_{b}" for fs in train_fs for b in BASES]

    rows_split: list[dict] = []
    rows_info: list[dict] = []
    frames: dict[str, list] = {e: [] for e in trained_ids}
    fitted_store: dict[str, dict] = {e: {} for e in trained_ids}

    for test_year in TEST_YEARS:
        train_end, val_end, max_date = fold_bounds(test_year, horizon)
        clim_k = clim_for_fold(train_end)
        vol_anom_k = anom_series(clim_k)
        kw = dict(WF_KW, horizon=horizon,
                  train_end=str(train_end.date()),
                  val_end=str(val_end.date()),
                  max_date=str(max_date.date()))
        aggs: dict[str, pd.DataFrame] = {}
        for fs in train_fs:
            feats = FEATURE_SETS[fs]
            e_pers = f"{fs}_pers"
            res_df, p_pers, fit_p = run_lstm_multi_seed(
                datasets_total, eq_total["vol"], feature_cols=feats,
                target_mode="delta",
                experiment_name=f"{e_pers}_h{horizon}_{test_year}", **kw)
            _collect(e_pers, test_year, res_df, p_pers, fit_p,
                     aggs, rows_split, rows_info, fitted_store)
            e_clim = f"{fs}_clim"
            res_df, p_anom, fit_c = run_lstm_multi_seed(
                datasets_total, vol_anom_k, feature_cols=feats,
                target_mode="level",
                experiment_name=f"{e_clim}_h{horizon}_{test_year}", **kw)
            p_clim = reconstruct_level(p_anom, clim_k)
            chk = p_clim.merge(
                vol_ct_total.rename(columns={"CT_TOTAL": "v"}),
                left_on="target_date", right_on="date", how="left")
            assert float((chk["y_true"] - chk["v"]).abs().max()) < 0.01
            if horizon == 12:
                chk2 = p_clim.merge(
                    vol_ct_total.rename(columns={"CT_TOTAL": "v"}),
                    left_on="issue_date", right_on="date", how="left")
                assert float((chk2["y_base"] - chk2["v"]).abs().max()) < 0.01
            _collect(e_clim, test_year, res_df, p_clim, fit_c,
                     aggs, rows_split, rows_info, fitted_store)
        for e in trained_ids:
            a = aggs[e].copy(); a["fold"] = test_year; frames[e].append(a)
        print(f"  [H={horizon}] fold {test_year} (treino {train_fs}) "
              f"concluído ({time.time()-t0:.0f}s)", flush=True)

    wf_exp = {e: pd.concat(fr, ignore_index=True) for e, fr in frames.items()}

    if reuse:
        wf_all = pd.read_csv(PRED_DIR / f"article_preds_wf_h{horizon}.csv")
        wf_all["target_date"] = pd.to_datetime(wf_all["target_date"])
        if trained_ids:
            _drop = ([f"y_{e}" for e in trained_ids]
                     + [f"y_{e}_std" for e in trained_ids])
            wf_all = wf_all.drop(columns=[c for c in _drop if c in wf_all.columns])
            ref = wf_exp[trained_ids[0]]
            d_cols = ref[["target_date", "y_true"]].copy()
            for e in trained_ids:
                assert wf_exp[e]["target_date"].equals(ref["target_date"])
                d_cols[f"y_{e}"] = wf_exp[e]["y_pred"].to_numpy()
                d_cols[f"y_{e}_std"] = wf_exp[e]["y_pred_std"].to_numpy()
            merged = wf_all.merge(d_cols, on="target_date", how="left",
                                  suffixes=("", "_chk"))
            assert np.allclose(merged["y_true"], merged["y_true_chk"], atol=0.01), \
                "y_true do run reaproveitado difere do recém-treinado"
            wf_all = merged.drop(columns=["y_true_chk"])
        _bf = pd.read_csv(METRICS_DIR / f"article_results_by_fold_h{horizon}.csv")
        _bf = _bf[~_bf["modelo"].isin([EXP_LABEL[e] for e in trained_ids])]
        by_fold = pd.concat([_bf,
            pd.DataFrame(_by_fold_rows(wf_all, trained_ids, horizon, False)),
        ], ignore_index=True)
        _tv = pd.read_csv(METRICS_DIR / f"article_train_val_by_fold_h{horizon}.csv")
        _tv = _tv[~_tv["exp"].isin(trained_ids)]
        train_val = pd.concat([_tv, pd.DataFrame(rows_split)], ignore_index=True)
        _ti = pd.read_csv(METRICS_DIR / f"article_train_info_by_fold_h{horizon}.csv")
        _ti = _ti[~_ti["exp"].isin(trained_ids)]
        train_info = pd.concat([_ti, pd.DataFrame(rows_info)], ignore_index=True)
        print(f"[H={horizon}] REAPROVEITADO do run anterior "
              f"(treinado agora: {train_fs or 'nada'}) "
              f"({time.time()-t0:.0f}s).", flush=True)
    else:
        ref = wf_exp[EXP_IDS[0]]
        for e in EXP_IDS[1:]:
            assert wf_exp[e]["target_date"].equals(ref["target_date"])
            assert np.allclose(wf_exp[e]["y_true"], ref["y_true"], atol=0.01)
        wf_all = ref[["target_date", "fold", "y_true", "y_base"]].rename(
            columns={"y_base": "y_base_persist"})
        for e in EXP_IDS:
            wf_all[f"y_{e}"] = wf_exp[e]["y_pred"].to_numpy()
            wf_all[f"y_{e}_std"] = wf_exp[e]["y_pred_std"].to_numpy()
        clim_by_fold_full = {y: clim_for_fold(fold_bounds(y, horizon)[0])
                             for y in TEST_YEARS}
        mtbf = _mean_train_by_fold(horizon)
        wf_all["y_base_clim"] = wf_all.apply(
            lambda r: clim_by_fold_full[r["fold"]][r["target_date"].month], axis=1)
        wf_all["y_base_mean_train"] = wf_all["fold"].map(mtbf)
        by_fold = pd.DataFrame(_by_fold_rows(wf_all, EXP_IDS, horizon, True))
        train_val = pd.DataFrame(rows_split)
        train_info = pd.DataFrame(rows_info)
        print(f"[H={horizon}] walk-forward COMPLETO em {time.time()-t0:.0f}s "
              f"({len(EXP_IDS)} exp × {len(TEST_YEARS)} folds × "
              f"{len(SEEDS_WF)} seeds).", flush=True)

    clim_by_fold = {y: clim_for_fold(fold_bounds(y, horizon)[0])
                    for y in TEST_YEARS}
    return {
        "horizon": horizon,
        "wf_all": wf_all,
        "by_fold": by_fold,
        "train_val": train_val,
        "train_info": train_info,
        "clim_by_fold": clim_by_fold,
        "fitted": fitted_store,
    }


In [ ]:
# Reaproveita do run anterior os experimentos JÁ presentes nos CSVs e treina
# apenas os ausentes. Nesta rodada A/B/C/D já existem -> não treina nada
# (mantém os resultados idênticos); só a QR/§10 é reajustada adiante.
def _existing_fs(h: int):
    p = PRED_DIR / f"article_preds_wf_h{h}.csv"
    if not p.exists():
        return None
    cols = pd.read_csv(p, nrows=1).columns
    return {fs for fs in FEATURE_SETS if f"y_{fs}_pers" in cols}


_ex = {h: _existing_fs(h) for h in HORIZONS}
if all(_ex[h] is not None for h in HORIZONS):
    _train_only = set(FEATURE_SETS) - set.intersection(*[_ex[h] for h in HORIZONS])
else:
    _train_only = None
print("Grade — treinar agora:",
      (_train_only or "nada (reaproveita tudo)") if _train_only is not None
      else "TODOS (sem run anterior)")
res_h3 = run_grid(3, train_only=_train_only)

In [ ]:
res_h12 = run_grid(12, train_only=_train_only)
res = {3: res_h3, 12: res_h12}

## § 7. Séries no período de teste — todos os modelos

Um painel por base de reconstrução (para não sobrecarregar), com os 3
feature sets + baselines; uma figura por horizonte. Linhas verticais =
fronteiras dos folds anuais. As predições são a média do ensemble de 10
seeds, concatenadas pelos 7 folds — uma série contínua out-of-sample de
84 meses (2019–2025).

In [ ]:
cores_fs = {"A": "#d97742", "B": "#7b5aa6", "C": "#2a9d8f", "D": "#c9a227"}


def plot_test_period(res_h: dict) -> None:
    # Article figure: 8 experiments + baselines on the walk-forward test.
    h = res_h["horizon"]
    w = res_h["wf_all"]
    dates = w["target_date"]
    fig, axes = plt.subplots(2, 1, figsize=(11.5, 8.2), sharex=True, sharey=True)
    for ax, base, titulo in [
        (axes[0], "pers",
         "(a) A/B/C/D \u2014 persistence reconstruction: V\u0302 = Vol(t) + \u0394\u0302"),
        (axes[1], "clim",
         "(b) A*/B*/C*/D* \u2014 climatology reconstruction: V\u0302 = Clim[m] + \u0394\u0302"),
    ]:
        ax.plot(dates, w["y_true"], color="#1f3a5f", lw=2.0,
                label="observed", zorder=7)
        for fs in FEATURE_SETS:
            e = f"{fs}_{base}"
            ax.plot(dates, w[f"y_{e}"], color=cores_fs[fs], lw=1.4,
                    label=EXP_LABEL[e], zorder=5)
        ax.plot(dates, w["y_base_persist"], color="#999999", lw=1.0,
                linestyle="--", label=f"persistence-{h}", zorder=3)
        ax.plot(dates, w["y_base_clim"], color="#4a7a4a", lw=1.1,
                linestyle="-.", label="climatology", zorder=3)
        for y in TEST_YEARS[:-1]:
            ax.axvline(pd.Timestamp(f"{y+1}-01-01"), color="gray", lw=0.5,
                       linestyle=":", alpha=0.5)
        ax.set_title(titulo, loc="left", fontweight="bold")
        ax.set_ylabel("CT_TOTAL volume (hm³)")
        ax.legend(loc="upper left", frameon=False, fontsize=8, ncol=3)
        ax.grid(True, alpha=0.25)
    axes[1].set_xlabel("target date")
    fig.suptitle(f"CT_TOTAL \u2014 H = {h} months: 8 experiments vs baselines "
                 "(walk-forward test 2019\u20132025)",
                 fontweight="bold", x=0.02, ha="left", y=1.0)
    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"ct_total_lstm_h{h}_test_period.png", dpi=300)
    plt.show()


for h in HORIZONS:
    plot_test_period(res[h])

### 7.1. Features observadas no período de teste (2019–2025)

Painel das variáveis de entrada no período de walk-forward, com as fronteiras
dos folds anuais (linhas pontilhadas). Serve para ler as séries do §7 contra o
que o modelo "via": a recessão de 2021–2022, o reenchimento abrupto de 2023 e o
deplecionamento de 2025. Migrado do antigo `predict_lstm` (§9.1), agora no
notebook do artigo. `pr_RollingMean12` é só referência de tendência — não entra
em nenhum feature set.

In [ ]:
feat_df = datasets_total["CT_TOTAL"].copy()
_wf_ref = res[12]["wf_all"]                      # mesmo período-teste nos dois H
t0, t1 = _wf_ref["target_date"].min(), _wf_ref["target_date"].max()
fwin = feat_df[(feat_df["date"] >= t0) & (feat_df["date"] <= t1)].copy()
vwin = vol_ct_total[
    (vol_ct_total["date"] >= t0) & (vol_ct_total["date"] <= t1)].copy()
clim_ref = res[12]["clim_by_fold"][TEST_YEARS[-1]]   # referência visual: último fold
vwin["clim"] = vwin["date"].dt.month.map(clim_ref)

fig, axes = plt.subplots(4, 2, figsize=(12.5, 11), sharex=True)


def _mark_folds(ax):
    for y in TEST_YEARS[:-1]:
        ax.axvline(pd.Timestamp(f"{y+1}-01-01"), color="gray", lw=0.5,
                   linestyle=":", alpha=0.5)


ax = axes[0, 0]
ax.plot(fwin["date"], fwin["pr"], color="#1f3a5f", lw=1.0, alpha=0.75, label="pr")
ax.plot(fwin["date"], fwin["pr_RollingMean12"], color="#d97742", lw=1.8,
        label="pr_RollingMean12*")
ax.set_title("(a) Precipitação (mm/mês)", loc="left", fontweight="bold")
ax.legend(frameon=False, fontsize=8)

ax = axes[0, 1]
ax.plot(fwin["date"], fwin["eto"], color="#1f3a5f", lw=1.0, alpha=0.75, label="eto")
ax.plot(fwin["date"], fwin["eto_RollingMean12"], color="#d97742", lw=1.8,
        label="eto_RollingMean12")
ax.set_title("(b) Evapotranspiração de referência (mm/mês)",
             loc="left", fontweight="bold")
ax.legend(frameon=False, fontsize=8)

ax = axes[1, 0]
ax.plot(fwin["date"], fwin["q"], color="#1f3a5f", lw=1.0, alpha=0.75, label="q")
ax.plot(fwin["date"], fwin["q_RollingMean6"], color="#d97742", lw=1.8,
        label="q_RollingMean6")
ax.set_title("(c) Vazão natural (m³/s)", loc="left", fontweight="bold")
ax.legend(frameon=False, fontsize=8)

ax = axes[1, 1]
ax.plot(fwin["date"], fwin["SRI12"], color="#1f3a5f", lw=1.5)
ax.axhline(0, color="gray", lw=0.7, linestyle="--")
ax.axhline(-1, color="#c44e52", lw=0.7, linestyle=":")
ax.set_title("(d) SRI12 — índice padronizado de vazão (12 meses)",
             loc="left", fontweight="bold")

ax = axes[2, 0]
ax.plot(vwin["date"], vwin["CT_TOTAL"], color="#1f3a5f", lw=1.6, label="Vol (nível)")
ax.plot(vwin["date"], vwin["clim"], color="#4a7a4a", lw=1.2,
        linestyle="-.", label="Clim[mês] (ref. último fold)")
ax.set_title("(e) Volume CT_TOTAL (hm³) — Vol_t dos experimentos base persistência",
             loc="left", fontweight="bold")
ax.legend(frameon=False, fontsize=8)

ax = axes[2, 1]
ax.plot(vwin["date"], vwin["CT_TOTAL"] - vwin["clim"], color="#7b5aa6", lw=1.6)
ax.axhline(0, color="gray", lw=0.7, linestyle="--")
ax.set_title("(f) Anomalia de volume (hm³) — Vol_t dos experimentos base clim.",
             loc="left", fontweight="bold")

ax = axes[3, 0]
for col, cor in [("qout", "#1f3a5f"), ("qbal", "#d97742"),
                 ("qpbs_net", "#4a7a4a"), ("qelev", "#7b5aa6")]:
    ax.plot(fwin["date"], fwin[col], lw=1.1, color=cor, label=col)
ax.set_title("(g) Operacionais (m³/s) — qbal (sets C/D) e roteamento (set B)",
             loc="left", fontweight="bold")
ax.legend(frameon=False, fontsize=8, ncol=2)
ax.set_xlabel("data")

ax = axes[3, 1]
ax.plot(fwin["date"], fwin["month_sin"], color="#1f3a5f", lw=1.1, label="month_sin")
ax.plot(fwin["date"], fwin["month_cos"], color="#d97742", lw=1.1, label="month_cos")
ax.fill_between(fwin["date"], -1, fwin["is_dry"] * 2 - 1, step="mid",
                color="#d97742", alpha=0.12, label="is_dry")
ax.set_title("(h) Calendário — month_sin/cos + is_dry (sombreado)",
             loc="left", fontweight="bold")
ax.legend(frameon=False, fontsize=8)
ax.set_xlabel("data")

for a in axes.ravel():
    a.grid(True, alpha=0.25)
    _mark_folds(a)

fig.suptitle("Features no período de walk-forward "
             f"({t0.date()} a {t1.date()}) — CT_TOTAL\n"
             "*pr_RollingMean12 não entra nos modelos; referência de tendência.",
             fontweight="bold", x=0.02, ha="left", y=1.005)
fig.tight_layout()
fig.savefig(FIG_DIR / "ct_total_lstm_features_test_period.png",
            dpi=300, bbox_inches="tight")
plt.show()


## § 8. Tabelas de performance — teste (pooled) + treinamento

Métricas idênticas às do `lstm_exploratory` (Parte II) (fundamentação e referências na
§13), calculadas na série de teste pooled (84 meses):

1. **Contínuas**: RMSE, MAE, r (Pearson), NSE.
2. **Decomposição KGE** (Gupta et al., 2009): $r$, $\alpha$
   (variabilidade — $\alpha<1$ = previsões suaves demais), $\beta$ (viés).
3. **Detecção categórica de secas** (Wilks, 2019) com limiar variável
   (Van Loon, 2015): mês de seca = volume < quantil 20% do TREINO do
   fold, por mês-calendário → POD/FAR/CSI/HSS.
4. **Discriminação**: AUC (Mann–Whitney); **erros condicionais**:
   RMSE_seca e ME_seca (>0 = falsa segurança).

**Coeficientes do treinamento** (colunas adicionais, só experimentos):
RMSE/NSE/r do split de treino, RMSE da validação e época ótima — médias
entre os 7 folds da predição ensemble (o teste é pooled; treino/val são
"média entre folds" porque cada fold retreina com histórico diferente).
Servem de diagnóstico de sub/superajuste: comparar RMSE_treino ×
RMSE_val × RMSE (teste).

In [ ]:
# ---------- Funções de avaliação (idênticas ao lstm_exploratory, Parte II §8.2) ----------
def kge_decomposition(y_true_arr, y_pred_arr) -> dict:
    # KGE de Gupta et al. (2009) e componentes r / alpha / beta.
    o = np.asarray(y_true_arr, float)
    p = np.asarray(y_pred_arr, float)
    r = float(np.corrcoef(o, p)[0, 1])
    alpha = float(np.std(p) / np.std(o))
    beta = float(np.mean(p) / np.mean(o))
    return {"kge": 1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2
                               + (beta - 1) ** 2),
            "r": r, "alpha": alpha, "beta": beta}


def drought_contingency(obs_event, pred_event) -> dict:
    # Tabela de contingência 2x2 e escores categóricos (Wilks, 2019).
    obs_event = np.asarray(obs_event, bool)
    pred_event = np.asarray(pred_event, bool)
    a = int((obs_event & pred_event).sum())      # acertos (hits)
    b = int((~obs_event & pred_event).sum())     # alarmes falsos
    c = int((obs_event & ~pred_event).sum())     # secas perdidas
    d = int((~obs_event & ~pred_event).sum())    # negativos corretos
    n = a + b + c + d
    esperado = ((a + b) * (a + c) + (c + d) * (b + d)) / n
    return {
        "POD": a / (a + c) if a + c else np.nan,
        "FAR": b / (a + b) if a + b else np.nan,
        "CSI": a / (a + b + c) if a + b + c else np.nan,
        "HSS": (a + d - esperado) / (n - esperado) if n != esperado else np.nan,
        "hits": a, "misses": c, "false_alarms": b,
    }


def roc_auc_low(score, event) -> float:
    # AUC p/ eventos de MÍNIMO: P(score de mês de seca < score de mês
    # normal); equivalente a Mann-Whitney U normalizado.
    s = np.asarray(score, float)
    e = np.asarray(event, bool)
    sd, sn = s[e], s[~e]
    if len(sd) == 0 or len(sn) == 0:
        return float("nan")
    vitorias = ((sd[:, None] < sn[None, :]).sum()
                + 0.5 * (sd[:, None] == sn[None, :]).sum())
    return float(vitorias / (len(sd) * len(sn)))


def monthly_threshold(train_end: pd.Timestamp, q: float) -> pd.Series:
    # Limiar variável (Van Loon, 2015): quantil q por mês-calendário,
    # calculado APENAS no treino do fold (sem vazamento).
    tw = (vol_ct_total["date"] >= MIN_DATE) & (vol_ct_total["date"] <= train_end)
    return (vol_ct_total.loc[tw].assign(m=lambda d: d["date"].dt.month)
            .groupby("m")["CT_TOTAL"].quantile(q))


DROUGHT_Q = 0.20   # Q20 = 80% de excedência (Van Loon 2015: 70-90%)

# train_end de cada fold NÃO depende do horizonte -> limiares únicos.
thr20 = {y: monthly_threshold(fold_bounds(y, 12)[0], DROUGHT_Q)
         for y in TEST_YEARS}
thr30 = {y: monthly_threshold(fold_bounds(y, 12)[0], 0.30)
         for y in TEST_YEARS}

for h in HORIZONS:
    w = res[h]["wf_all"]
    w["thr20"] = w.apply(
        lambda r: thr20[r["fold"]][r["target_date"].month], axis=1)
    w["thr30"] = w.apply(
        lambda r: thr30[r["fold"]][r["target_date"].month], axis=1)
    w["obs_seca20"] = w["y_true"] < w["thr20"]
    w["obs_seca30"] = w["y_true"] < w["thr30"]

# Os alvos observados são os mesmos nos dois horizontes (2019-01..2025-12).
w = res[3]["wf_all"]
print(f"Meses de seca observada (Q20 mensal, por fold): "
      f"{int(w['obs_seca20'].sum())}/{len(w)}")
print(w.groupby('fold')['obs_seca20'].sum().to_string())
print(f"\nCom Q30: {int(w['obs_seca30'].sum())}/{len(w)} meses")

In [ ]:
def predictors_for(h: int) -> dict[str, np.ndarray]:
    # Dicionário rótulo -> série prevista (pooled) para um horizonte.
    w = res[h]["wf_all"]
    d = {EXP_LABEL[e]: w[f"y_{e}"].to_numpy() for e in EXP_IDS}
    d[f"persistência-{h}"] = w["y_base_persist"].to_numpy()
    d["média treino"] = w["y_base_mean_train"].to_numpy()
    d["climatologia"] = w["y_base_clim"].to_numpy()
    return d


def build_perf_table(h: int) -> pd.DataFrame:
    # Tabela completa da §8: métricas de teste (pooled) + treinamento.
    w = res[h]["wf_all"]
    y = w["y_true"].to_numpy()
    seca = w["obs_seca20"].to_numpy()
    tv = res[h]["train_val"]
    info = res[h]["train_info"]

    linhas = []
    for nome, p in predictors_for(h).items():
        m = compute_regression_metrics(y, p)
        k = kge_decomposition(y, p)
        ct = drought_contingency(w["obs_seca20"], p < w["thr20"])
        linhas.append({
            "preditor": nome,
            "RMSE": m["rmse"], "MAE": m["mae"], "r": m["corr"],
            "NSE": m["nse"],
            "KGE": k["kge"], "alpha": k["alpha"], "beta": k["beta"],
            "POD_q20": ct["POD"], "FAR_q20": ct["FAR"],
            "CSI_q20": ct["CSI"], "HSS_q20": ct["HSS"],
            "AUC": roc_auc_low(p, seca),
            "RMSE_seca": float(np.sqrt(np.mean((y[seca] - p[seca]) ** 2))),
            "ME_seca": float(np.mean(p[seca] - y[seca])),
        })
    tab = pd.DataFrame(linhas).set_index("preditor")

    # Coeficientes do TREINAMENTO (média entre folds; só experimentos).
    for e in EXP_IDS:
        lab = EXP_LABEL[e]
        tr = tv[(tv["exp"] == e) & (tv["split"] == "train")]
        va = tv[(tv["exp"] == e) & (tv["split"] == "val")]
        tab.loc[lab, "RMSE_treino"] = tr["rmse"].mean()
        tab.loc[lab, "NSE_treino"] = tr["nse"].mean()
        tab.loc[lab, "r_treino"] = tr["corr"].mean()
        tab.loc[lab, "RMSE_val"] = va["rmse"].mean()
        tab.loc[lab, "época_ótima"] = info.loc[
            info["exp"] == e, "best_epoch"].mean()
    return tab.round(3)


perf = {h: build_perf_table(h) for h in HORIZONS}
print("H = 3 meses — teste pooled 2019–2025 (84 meses):")
perf[3]

In [ ]:
print("H = 12 meses — teste pooled 2019–2025 (84 meses):")
perf[12]

In [ ]:
# Robustez entre folds: RMSE (média/desvio/amplitude) por preditor.
resumo_folds = {}
for h in HORIZONS:
    bf = res[h]["by_fold"]
    r = bf.groupby("modelo")["rmse"].agg(["mean", "std", "min", "max"]).round(1)
    r["amplitude"] = r["max"] - r["min"]
    ordem = ([EXP_LABEL[e] for e in EXP_IDS]
             + [f"persistência-{h}", "média treino", "climatologia"])
    resumo_folds[h] = r.reindex(ordem)

print("H = 3 — RMSE por fold (resumo):")
display(resumo_folds[3])
print("H = 12 — RMSE por fold (resumo):")
display(resumo_folds[12])

### 8.1. Leitura

**H = 3 — `C` (A + qbal) lidera; `SRI12` não ajuda no curto prazo.** C tem o melhor RMSE (52,6), o melhor KGE (0,940) e a melhor detecção de secas (POD 0,750, HSS 0,691, AUC 0,903). Acrescentar `SRI12` (→ D) **piora** em H=3 (RMSE 55,7, POD 0,625): o índice de 12 meses não traz sinal útil no horizonte curto. Ordem por RMSE: C (52,6) < C* (55,0) < D (55,7) < A (56,8).

**H = 12 — `D` (A + qbal + SRI12) lidera; `SRI12` ajuda no longo prazo.** No horizonte anual, D **detecta muito mais secas** (POD 0,562 vs 0,438 de A e 0,375 de C), com o melhor HSS (0,366), a melhor AUC (0,854) e **muito menos viés de seca** (ME_seca 29,0 vs 51,8 de A, 47,6 de C) — ainda que A tenha RMSE marginalmente melhor (132,0 vs 135,7). A memória longa do `SRI12` é o que agrega no horizonte de um ano.

**O operacional COMPLETO (`B`) continua o pior** nos dois horizontes — confirma a lição: **uma** variável operacional (o balanço agregado `qbal`) ajuda; a **rota completa** (B) atrapalha.

**Base climatologia (`*`) subdispersa** (α cai). `C*` fica competitivo em H=3 (RMSE 55,0); `D*` tem o 2º melhor RMSE em H=12 (133,8), mas detecção inferior à de D.

**Robustez**: C é o mais estável em H=3 (amplitude entre folds ≈ 48 hm³ vs 72 de A). 2023 permanece o fold mais difícil nos dois horizontes.

### 8.2. RMSE por fold — a escolha do ano-teste importa?

RMSE anual de cada experimento e dos baselines nos 7 folds, para os dois
horizontes (cor = feature set; tracejado = base persistência, sólido = base
climatologia). Mostra o que a média pooled esconde: em H = 3 o experimento
selecionado (C) fica abaixo da persistência em todos os anos; em H = 12 o fold de 2023
(reenchimento abrupto) domina o erro de todos os modelos. Migrado do antigo
`predict_lstm` (§7), agora para H = 3 e H = 12 e com o set D.

In [ ]:
cores_fs = {"A": "#d97742", "B": "#7b5aa6", "C": "#2a9d8f", "D": "#c9a227"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, h in zip(axes, HORIZONS):
    pv = res[h]["by_fold"].pivot_table(index="fold", columns="modelo",
                                       values="rmse")
    for e in EXP_IDS:
        fs, base = e.split("_")
        lab = EXP_LABEL[e]
        ax.plot(pv.index, pv[lab], "--s" if base == "pers" else "-o",
                color=cores_fs[fs], lw=1.5, ms=4.5, label=lab)
    ax.plot(pv.index, pv["climatologia"], "-.^", color="#4a7a4a", lw=1.3,
            ms=5, label="climatologia")
    ax.plot(pv.index, pv[f"persistência-{h}"], ":v", color="#999999", lw=1.3,
            ms=5, label=f"persistência-{h}")
    ax.set_title(f"H = {h} meses", loc="left", fontweight="bold")
    ax.set_xlabel("ano-teste (fold)")
    ax.set_xticks(TEST_YEARS)
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel("RMSE (hm³)")
axes[1].legend(loc="upper left", frameon=False, fontsize=8, ncol=2)
fig.suptitle("RMSE por fold — 8 experimentos e baselines",
             fontweight="bold", x=0.02, ha="left")
fig.tight_layout()
fig.savefig(FIG_DIR / "ct_total_lstm_rmse_by_fold_h3_h12.png",
            dpi=300, bbox_inches="tight")
plt.show()


## § 9. Seleção do melhor experimento por horizonte — ranking multi-métrica

Mesmo critério transparente do `lstm_exploratory` (Parte II) (§10): ranking composto
sobre 7 métricas com pesos iguais — RMSE (menor melhor), KGE, POD_q20,
HSS_q20, AUC (maiores melhor), FAR_q20 e |ME_seca| (menores melhor;
experimento que nunca alerta recebe FAR = 1, o pior caso). O menor rank
médio de cada horizonte é promovido à regressão quantílica da §10.

Limitações assumidas: pesos iguais são uma escolha (não uma verdade);
as métricas categóricas têm incerteza alta com poucos eventos; e
selecionar no teste é uma forma branda de ajuste — o selecionado é "o
melhor candidato atual", não vencedor definitivo.

In [ ]:
rankings: dict[int, pd.DataFrame] = {}
BEST: dict[int, str] = {}
for h in HORIZONS:
    t = perf[h].loc[[EXP_LABEL[e] for e in EXP_IDS]]
    criterios = pd.DataFrame({
        "RMSE": t["RMSE"],
        "KGE": t["KGE"],
        "POD_q20": t["POD_q20"],
        "FAR_q20": t["FAR_q20"].fillna(1.0),   # nunca alertar = pior FAR
        "HSS_q20": t["HSS_q20"],
        "AUC": t["AUC"],
        "abs_ME_seca": t["ME_seca"].abs(),
    })
    ASCENDENTE = {"RMSE": True, "FAR_q20": True, "abs_ME_seca": True}
    ranks = pd.DataFrame({
        c: criterios[c].rank(ascending=ASCENDENTE.get(c, False))
        for c in criterios.columns
    })
    criterios["rank_médio"] = ranks.mean(axis=1).round(2)
    criterios = criterios.sort_values("rank_médio").round(3)
    rankings[h] = criterios
    BEST[h] = LABEL2ID[criterios.index[0]]
    print(f"H={h:>2}: melhor experimento (rank médio) = "
          f"{criterios.index[0]}  (id interno: {BEST[h]})")

print("\nH = 3:")
display(rankings[3])
print("H = 12:")
display(rankings[12])

### 9.1. Leitura da seleção

**Vencedores por horizonte: `C` (H=3, rank médio 1,36) e `D` (H=12, rank 1,71).** O `SRI12` inverte de papel com o horizonte.

**Em H=3, `C` vence por consenso** — melhor ou quase-melhor em todos os critérios (RMSE 52,6, KGE 0,94, POD 0,75, HSS 0,69, AUC 0,90, |ME_seca| 14,3). `D` (com SRI12) cai para 3º (rank 3,21), atrás de A — SRI12 não ajuda no curto prazo.

**Em H=12, `D` vence por composição** — A tem RMSE ligeiramente melhor (132,0 vs 135,7), mas D domina em detecção (POD 0,562, HSS 0,366), discriminação (AUC 0,854) e viés de seca (ME_seca 29,0). É a mesma tensão RMSE-vs-detecção-de-seca, resolvida pelo ranking multi-métrica a favor de `D`.

**Promovidos à regressão quantílica (§10)**: `C` em H=3 e `D` em H=12 — refletindo que o `SRI12` só agrega no horizonte longo.

### 9.2. Significância da melhora — teste de Diebold–Mariano

Para cada horizonte, o erro mensal do experimento selecionado é comparado com o
da persistência e o da climatologia na série walk-forward agregada (84 meses),
sob perda quadrática. A variância do diferencial de perda inclui
autocovariâncias até a defasagem H − 1 (previsões sobrepostas), com a correção
de amostra pequena de Harvey et al. (1997) e t de Student com n − 1 graus de
liberdade. Diferencial médio negativo indica que o modelo erra menos que a
referência. Mesma função do `xgboost_article.ipynb` (§3.2):
`Previsao.Scripts.forecast_evaluation.diebold_mariano_test`. O artigo reporta
H = 3 contra a persistência.


In [ ]:
dm_rows = []
for h in HORIZONS:
    w = res[h]["wf_all"].sort_values("target_date")
    for nome, col in [(f"persistência-{h}", "y_base_persist"),
                      ("climatologia", "y_base_clim")]:
        dm_rows.append({
            "H": h, "modelo": EXP_LABEL[BEST[h]], "referência": nome,
            **diebold_mariano_test(w["y_true"].to_numpy(),
                                   w[f"y_{BEST[h]}"].to_numpy(),
                                   w[col].to_numpy(), horizon=h),
        })
dm_table = pd.DataFrame(dm_rows)
dm_table.round(4)

## § 10. Regressão quantílica dos melhores experimentos — previsão probabilística

Mesmo modelo com head de **7 saídas** (quantis 0,05 / 0,10 / 0,25 / 0,50 /
0,75 / 0,90 / 0,95) e perda *pinball* (Koenker & Bassett, 1978; `PinballLoss`
do braço LSTM) — aprende a distribuição condicional. O **P10 é o gatilho de
alerta de seca** e a banda P10–P90 é um intervalo de 80 % **aprendido dos
dados**.

Protocolo idêntico ao walk-forward principal (7 folds, 10 seeds), com o
feature set/base do vencedor de cada horizonte (§9: **C** em H=3, **D** em
H=12). **Avaliação probabilística** no espírito de Quilty & Jahangir (2026):
**PICP** (cobertura empírica), **ACE** = PICP − PINC (desvio de calibração,
com sinal), **PIAW** (largura média = nitidez), **Winkler score** e o
**alpha index** (Renard et al., 2010 — confiabilidade agregada, ideal 1),
avaliados nas coberturas nominais PINC ∈ {0,50; 0,80; 0,90} e no *probability
plot*; complementados pela comparação de gatilhos **P50 (determinístico) ×
P10 (probabilístico)** (POD/FAR/CSI/HSS no limiar Q20).

In [ ]:
# Regressão quantílica com 7 quantis -> habilita PICP/ACE/PIAW por nível,
# diagrama de confiabilidade (probability plot) e alpha index (Renard 2010).
QLEVELS = (0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95)
QNAME = {q: f"q{int(round(q * 100)):02d}" for q in QLEVELS}
QCOL = {q: f"y_pred_{QNAME[q]}" for q in QLEVELS}
# Intervalos centrais simétricos -> PINC (coberturas nominais).
BANDS = {0.50: (0.25, 0.75), 0.80: (0.10, 0.90), 0.90: (0.05, 0.95)}


def reconstruct_level_qr(preds_qr: pd.DataFrame, clim: pd.Series) -> pd.DataFrame:
    # Anomalia -> nível físico: soma Clim[m(t+H)] a TODOS os quantis + y_true
    # + y_base (deslocamento aditivo preserva a ordem dos quantis).
    out = preds_qr.copy()
    out["target_date"] = pd.to_datetime(out["target_date"])
    ct = out["target_date"].dt.month.map(clim).to_numpy()
    cols = (["y_true", "y_base", "y_pred_p10", "y_pred_p50", "y_pred_p90"]
            + list(QCOL.values()))
    for col in cols:
        if col in out.columns:
            out[col] = preds_qr[col].to_numpy() + ct
    return out


wf_qr: dict[int, pd.DataFrame] = {}
qr_rows, trigger_rows, rel_rows, prob_rows = [], [], [], []
t_start_qr = time.time()

for h in HORIZONS:
    best_exp = BEST[h]
    best_fs, best_base = best_exp.rsplit("_", 1)
    frames_qr = []
    for test_year in TEST_YEARS:
        train_end, val_end, max_date = fold_bounds(test_year, h)
        clim_k = clim_for_fold(train_end)
        kw = dict(WF_KW, horizon=h, train_end=str(train_end.date()),
                  val_end=str(val_end.date()), max_date=str(max_date.date()))
        vol_input, tmode = ((eq_total["vol"], "delta") if best_base == "pers"
                            else (anom_series(clim_k), "level"))
        _, preds_qr, _ = run_lstm_quantile_multi_seed(
            datasets_total, vol_input, feature_cols=FEATURE_SETS[best_fs],
            target_mode=tmode, quantiles=QLEVELS,
            experiment_name=f"QR_{best_exp}_h{h}_{test_year}", **kw)
        if best_base == "clim":
            preds_qr = reconstruct_level_qr(preds_qr, clim_k)
        else:
            preds_qr = preds_qr.copy()
            preds_qr["target_date"] = pd.to_datetime(preds_qr["target_date"])
        test = preds_qr[preds_qr["split"] == "test"]
        spec = {"y_true": ("y_true", "first"), "y_base": ("y_base", "first")}
        for q in QLEVELS:
            spec[QNAME[q]] = (QCOL[q], "mean")     # ensemble entre seeds
        agg = (test.groupby("target_date").agg(**spec)
               .sort_index().reset_index())
        agg["fold"] = test_year
        frames_qr.append(agg)
        print(f"  [H={h}] QR fold {test_year} ({time.time()-t_start_qr:.0f}s)",
              flush=True)

    qq = pd.concat(frames_qr, ignore_index=True)
    qq["thr20"] = qq.apply(
        lambda r: thr20[r["fold"]][r["target_date"].month], axis=1)
    qq["obs_seca20"] = qq["y_true"] < qq["thr20"]
    assert qq["target_date"].equals(res[h]["wf_all"]["target_date"])
    assert np.allclose(qq["y_true"], res[h]["wf_all"]["y_true"], atol=0.01)
    wf_qr[h] = qq

    o = qq["y_true"].to_numpy()
    q50 = qq["q50"].to_numpy()
    m50 = compute_regression_metrics(o, q50)

    # PICP / ACE / PIAW / Winkler por PINC (banda central).
    for pinc, (lo_q, hi_q) in BANDS.items():
        lo, hi = qq[QNAME[lo_q]].to_numpy(), qq[QNAME[hi_q]].to_numpy()
        picp = coverage_rate(o, lo, hi)
        rel_rows.append({
            "H": h, "PINC": pinc, "PICP": picp, "ACE": picp - pinc,
            "PIAW (hm³)": mean_interval_width(lo, hi),
            "Winkler (hm³)": winkler_score(o, lo, hi, alpha=1 - pinc)})

    # Probability plot (confiabilidade) + alpha index (Renard et al., 2010).
    emp = np.array([float(np.mean(o <= qq[QNAME[q]].to_numpy())) for q in QLEVELS])
    for q, e in zip(QLEVELS, emp):
        prob_rows.append({"H": h, "nominal": q, "empirical": e})
    alpha_idx = 1.0 - 2.0 * float(np.mean(np.abs(emp - np.array(QLEVELS))))

    picp80 = coverage_rate(o, qq["q10"].to_numpy(), qq["q90"].to_numpy())
    qr_rows.append({
        "H": h, "experimento": EXP_LABEL[best_exp],
        "RMSE_P50": m50["rmse"], "KGE_P50": kge_decomposition(o, q50)["kge"],
        "PICP_80": picp80, "ACE_80": picp80 - 0.80,
        "PIAW_80 (hm³)": mean_interval_width(qq["q10"].to_numpy(),
                                             qq["q90"].to_numpy()),
        "Winkler_80 (hm³)": winkler_score(o, qq["q10"].to_numpy(),
                                          qq["q90"].to_numpy(), alpha=0.20),
        "alpha_index": alpha_idx})

    # Gatilhos de seca: P50 (determinístico) e P10 (probabilístico, q10).
    seca_qr = qq["obs_seca20"].to_numpy()
    for nome, gat in [("P50 < limiar (determinístico)", q50 < qq["thr20"]),
                      ("P10 < limiar (probabilístico)",
                       qq["q10"].to_numpy() < qq["thr20"])]:
        trigger_rows.append({"H": h, "experimento": EXP_LABEL[best_exp],
                             "gatilho": nome, **drought_contingency(seca_qr, gat)})

qr_summary = pd.DataFrame(qr_rows).set_index("H").round(3)
qr_triggers = pd.DataFrame(trigger_rows).set_index(["H", "gatilho"]).round(3)
qr_reliability = pd.DataFrame(rel_rows).set_index(["H", "PINC"]).round(3)
qr_probplot = pd.DataFrame(prob_rows)
print(f"\nQR (7 quantis) concluída em {time.time()-t_start_qr:.0f}s.")
display(qr_summary)
display(qr_reliability)

# ---- Figura (inglês): (a) reliability/probability plot + alpha; (b) sharpness
_cores = {3: "#2a9d8f", 12: "#7b5aa6"}
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.4))
ax = axes[0]
ax.plot([0, 1], [0, 1], color="0.4", lw=1.0, linestyle="--",
        label="perfect reliability (1:1)")
for h in HORIZONS:
    sub = qr_probplot[qr_probplot["H"] == h]
    a = float(qr_summary.loc[h, "alpha_index"])
    ax.plot(sub["nominal"], sub["empirical"], "o-", color=_cores[h], lw=1.6,
            ms=5, label=f"H={h} ({qr_summary.loc[h, 'experimento']}), \u03b1={a:.2f}")
ax.set_xlabel("nominal quantile level")
ax.set_ylabel("observed frequency P(y \u2264 Q)")
ax.set_title("(a) Reliability (probability plot)", loc="left", fontweight="bold")
ax.legend(loc="upper left", frameon=False, fontsize=8)
ax.grid(True, alpha=0.25)
ax = axes[1]
for h in HORIZONS:
    r = qr_reliability.loc[h]
    ax.plot(r.index.astype(float), r["PIAW (hm³)"], "o-", color=_cores[h],
            lw=1.6, ms=5, label=f"H={h} ({qr_summary.loc[h, 'experimento']})")
ax.set_xlabel("nominal coverage (PINC)")
ax.set_ylabel("mean interval width — PIAW (hm³)")
ax.set_title("(b) Sharpness", loc="left", fontweight="bold")
ax.legend(loc="upper left", frameon=False, fontsize=8)
ax.grid(True, alpha=0.25)
fig.suptitle("CT_TOTAL — quantile regression: reliability and sharpness "
             "(walk-forward test 2019\u20132025)",
             fontweight="bold", x=0.02, ha="left", y=1.02)
fig.tight_layout()
save_figure(fig, FIG_DIR / "ct_total_lstm_qr_reliability.png", dpi=300)
plt.show()

In [ ]:
# Article figure: 80% prediction bands (q10-q90) of the two winners.
fig, axes = plt.subplots(2, 1, figsize=(11.5, 9.2), sharex=True)
for ax, h, letra in [(axes[0], 3, "a"), (axes[1], 12, "b")]:
    q = wf_qr[h]
    w = res[h]["wf_all"]
    dq = q["target_date"]
    seca_q = q["obs_seca20"].to_numpy()
    ax.fill_between(dq, q["q10"], q["q90"], color="#2a9d8f",
                    alpha=0.20, lw=0, label="P10\u2013P90 band (80% nominal)")
    ax.plot(dq, q["q50"], color="#2a9d8f", lw=1.8,
            label=f"P50 \u2014 {EXP_LABEL[BEST[h]]} (QR)", zorder=5)
    ax.plot(dq, q["q10"], color="#2a9d8f", lw=0.9, linestyle=":",
            label="P10 (drought trigger)", zorder=5)
    ax.plot(dq, q["y_true"], color="#1f3a5f", lw=2.1, label="observed", zorder=7)
    ax.plot(dq, w["y_base_clim"], color="#4a7a4a", lw=1.2,
            linestyle="-.", label="climatology (expanding)", zorder=3)
    ax.plot(dq, w["y_base_persist"], color="#999999", lw=1.0,
            linestyle="--", label=f"persistence-{h}", zorder=3)
    ax.plot(dq, q["thr20"], color="#c44e52", lw=1.0, linestyle=":",
            label="drought threshold (monthly Q20)", zorder=4)
    ax.scatter(dq[seca_q], q["y_true"][seca_q], marker="v", s=42,
               color="#c44e52", zorder=8, label="observed drought")
    for y in TEST_YEARS[:-1]:
        ax.axvline(pd.Timestamp(f"{y+1}-01-01"), color="gray", lw=0.5,
                   linestyle=":", alpha=0.5)
    ax.set_title(f"({letra}) H = {h} months \u2014 {EXP_LABEL[BEST[h]]} "
                 "(quantile regression)", loc="left", fontweight="bold")
    ax.set_ylabel("CT_TOTAL volume (hm³)")
    ax.legend(loc="upper left", frameon=False, fontsize=8, ncol=2)
    ax.grid(True, alpha=0.25)
axes[1].set_xlabel("target date")
fig.suptitle("CT_TOTAL \u2014 P10\u2013P90 prediction bands of the best "
             "experiments (walk-forward 2019\u20132025)",
             fontweight="bold", x=0.02, ha="left", y=1.0)
fig.tight_layout()
save_figure(fig, FIG_DIR / "ct_total_lstm_qr_best_h3_h12.png", dpi=300)
plt.show()

### 10.1. Leitura — confiabilidade, nitidez e gatilhos

**Ambos os vencedores são bem calibrados, mas ligeiramente superconfiantes.**
O *alpha index* é alto (**0,91** em H=3 / **0,92** em H=12) e o *probability
plot* fica colado à reta 1:1. Mas o **ACE é negativo em todos os PINC**: a
banda de 80 % cobre 73,8 % em H=3 (ACE −0,062) e 77,4 % em H=12 (ACE −0,026)
— os intervalos são **um pouco estreitos demais** (leve subcobertura / excesso
de confiança), efeito mais forte em H=3 e nos PINC altos. Leitura honesta: o
modelo é confiável na estrutura, mas subestima levemente a incerteza.

**A nitidez cresce com o horizonte, como esperado.** A largura média (PIAW) da
banda de 80 % é **124 hm³** em H=3 (C) contra **293 hm³** em H=12 (D), e o
Winkler acompanha (196 → 590 hm³) — o horizonte anual é intrinsecamente mais
incerto.

**P50 da QR × previsão pontual.** Em H=12 o P50 quantílico **detecta muito mal
as secas** (POD 0,062, HSS −0,10, 1/16) e não supera o D pontual (RMSE_P50
137,3 ≈ 135,7). Em H=3 o P50 da QR é forte (POD 0,812, HSS 0,736, 13/16).
**Recomendação**: em H=12, usar o D pontual como previsão central e a QR
**apenas pela camada de alerta P10**.

**Gatilho P50 × P10.** Em H=3 (C) o P50 já detecta 13/16 secas (HSS 0,74, 4
alarmes falsos); subir para o P10 fecha as 3 restantes (POD 1,0) mas o HSS cai
a 0,60 (14 alarmes falsos) — o ganho de sensibilidade tem custo. Em H=12 (D) o
P50 quase não alerta (1/16); o **P10 é indispensável**, detectando 15/16 (POD
0,938, HSS 0,245) ao custo de 34 alarmes falsos — troca defensável quando
perder uma seca custa mais que investigar um alarme.

## § 11. Interpretabilidade (xAI) — qual mês da janela e qual feature informam a previsão

Análogos sequenciais ao SHAP/PDP do braço XGBoost, herdados do
`lstm_exploratory.ipynb` (Parte I) (§15) — o input da LSTM é 3-D
($n \times W \times p$), então as atribuições são mapas
(timestep × feature):

| Técnica | Análogo XGB | O que mede | Saída |
|---|---|---|---|
| Saliência temporal | — | $\|\partial\hat y/\partial X\|$ por célula da janela | $(W, p)$ |
| Sensibilidade à perturbação | ICE (ablação) | $\|\Delta\hat y\|$ ao substituir $X[:, t, f]$ pela média | $(W, p)$ |
| Seq-PDP | PDP | $E[\hat y \mid X_{t,f}=v]$ sobre grelha de percentis | $(W, p, \text{grid})$ |

**Protocolo desta seção** (específico do walk-forward): saliência e
perturbação são calculadas para o **experimento vencedor de cada
horizonte** (§9), no modelo da seed 42 de **cada um dos 7 folds**,
sobre as janelas de teste do próprio fold, e **médias entre folds** —
o mapa resultante representa o período de teste inteiro (84 janelas),
não um único ajuste. Os inputs estão na escala padronizada de cada
fold (z-scores), o que torna as magnitudes comparáveis entre features.

O Seq-PDP (mais caro) é calculado no modelo do **último fold (2025)** —
o mais próximo do uso operacional — para as **4 features mais
importantes** pela perturbação, com eixos convertidos de volta às
**unidades físicas** via o scaler do fold.

**Leitura não causal** (regra homóloga ao SHAP, `docs/methodological_rules.md` §3.6):
os mapas indicam *sensibilidade do modelo*, não efeito físico — usar
"indica", "atua como proxy de", "está associado a".

In [ ]:
# Garante os modelos seed-42 do vencedor para o xAI. No modo incremental,
# se o vencedor foi reaproveitado (não treinado nesta rodada), treina só a
# seed 42 dos 7 folds (barato) — o suficiente para saliência/perturbação.
def _ensure_fitted_seed42(h: int, exp_id: str) -> None:
    store = res[h]["fitted"]
    if store.get(exp_id):
        return
    fs, base = exp_id.rsplit("_", 1)
    store[exp_id] = {}
    for test_year in TEST_YEARS:
        train_end, val_end, max_date = fold_bounds(test_year, h)
        clim_k = clim_for_fold(train_end)
        kw = dict(WF_KW, horizon=h, seeds=[42],
                  train_end=str(train_end.date()),
                  val_end=str(val_end.date()),
                  max_date=str(max_date.date()))
        vol_in, tmode = ((eq_total["vol"], "delta") if base == "pers"
                         else (anom_series(clim_k), "level"))
        _, _, fit_list = run_lstm_multi_seed(
            datasets_total, vol_in, feature_cols=FEATURE_SETS[fs],
            target_mode=tmode,
            experiment_name=f"xai_{exp_id}_h{h}_{test_year}", **kw)
        store[exp_id][test_year] = _keep_fitted(fit_list[0])

for _h in HORIZONS:
    _ensure_fitted_seed42(_h, BEST[_h])

XAI: dict[int, dict] = {}
for h in HORIZONS:
    e = BEST[h]
    lab = EXP_LABEL[e]
    sal_list, pert_list, feats = [], [], None
    for y_fold in TEST_YEARS:
        f = res[h]["fitted"][e][y_fold]
        feats = f["feature_cols"]
        sal_list.append(compute_temporal_saliency(
            f["model"], f["X_test"], feature_names=feats))
        pert_list.append(compute_perturbation_sensitivity(
            f["model"], f["X_test"], feature_names=feats))
    sal = np.mean(sal_list, axis=0)
    pert = np.mean(pert_list, axis=0)
    XAI[h] = {"sal": sal, "pert": pert, "features": feats}

    plot_xai_heatmap(
        sal, feature_names=feats,
        title=f"CT_TOTAL H={h} — {lab}: temporal saliency |∂ŷ/∂X| "
              "(mean of 7 folds, seed 42)",
        ylabel="window month (t−11 oldest → t−0 latest)",
        output_path=XAI_DIR / f"xai_saliency_best_h{h}.png")
    plot_xai_heatmap(
        pert, feature_names=feats,
        title=f"CT_TOTAL H={h} — {lab}: perturbation sensitivity |Δŷ| "
              "(mean of 7 folds, seed 42)",
        ylabel="window month (t−11 oldest → t−0 latest)",
        output_path=XAI_DIR / f"xai_perturbation_best_h{h}.png")

    imp = pd.Series(pert.mean(axis=0), index=feats).sort_values(ascending=False)
    print(f"\nH={h} ({lab}) — importância por perturbação "
          "(média sobre a janela):")
    print(imp.round(4).to_string())
plt.show()

In [ ]:
# Seq-PDP das 4 features mais importantes — modelo do último fold (2025),
# eixos em unidades físicas (inversão do scaler do fold).
SEQPDP_TIMESTEPS = [0, 5, 8, 11]   # t-11, t-6, t-3, t-0
W_LEN = 12

for h in HORIZONS:
    e = BEST[h]
    lab = EXP_LABEL[e]
    base = e.split("_")[1]
    f = res[h]["fitted"][e][TEST_YEARS[-1]]
    model, X, feats = f["model"], f["X_test"], f["feature_cols"]
    fscaler = f["scaler"].feature_scaler
    tscaler = f["scaler"].target_scaler

    imp = XAI[h]["pert"].mean(axis=0)
    top_idx = [int(i) for i in np.argsort(imp)[::-1][:4]]
    print(f"H={h} ({lab}) — Seq-PDP para: {[feats[i] for i in top_idx]}")

    pdp_vals, grids = compute_seq_pdp(
        model, X, feature_names=feats, grid_resolution=30)

    ylab = ("Δ̂Vol at t+H (hm³)" if base == "pers"
            else "predicted anomaly at t+H (hm³)")
    fig, axes = plt.subplots(len(SEQPDP_TIMESTEPS), len(top_idx),
                             figsize=(11.5, 9.0), sharex="col")
    cores_t = plt.cm.Blues(np.linspace(0.45, 0.95, len(SEQPDP_TIMESTEPS)))
    for i, t in enumerate(SEQPDP_TIMESTEPS):
        for j, fi in enumerate(top_idx):
            ax = axes[i][j]
            x_phys = grids[t][fi] * fscaler.scale_[fi] + fscaler.mean_[fi]
            y_phys = tscaler.inverse_transform(
                pdp_vals[t, fi, :].reshape(-1, 1)).ravel()
            ax.plot(x_phys, y_phys, color=cores_t[i], lw=1.8)
            ax.axhline(y_phys.mean(), color="grey", lw=0.7,
                       linestyle="--", alpha=0.6)
            if i == 0:
                ax.set_title(feats[fi], fontsize=9, fontweight="bold")
            if j == 0:
                ax.set_ylabel(f"t−{W_LEN - 1 - t}\n{ylab}", fontsize=8)
            ax.tick_params(labelsize=7)
            ax.grid(True, alpha=0.2)
    fig.suptitle(f"CT_TOTAL H={h} — {lab}: Seq-PDP of the 4 most important "
                 "features (fold 2025, physical units)",
                 fontweight="bold", x=0.02, ha="left", y=1.0)
    fig.tight_layout()
    save_figure(fig, XAI_DIR / f"xai_seqpdp_best_h{h}.png", dpi=300)
plt.show()

### 11.1. Leitura da interpretabilidade

**Inversão pr ↔ eto e o protagonismo de `qbal`.** No vencedor de H=3 (`C`), o balanço **`qbal` é o driver dominante** (topo por saliência e por perturbação), seguido de `pr` e do calendário — a rede recorrente confirma o sinal de fluxo do `qbal` que o SHAP do XGBoost apontou. Em H=12 (vencedor `D`) a **`eto` assume o topo** por perturbação (0,048), com `qbal` em 2º e `Vol_t` no topo por saliência: no horizonte anual pesam o regime (eto), o estado de armazenamento (Vol_t) e o balanço (qbal). Nota honesta: o ganho do `SRI12` em H=12 aparece na **detecção de secas** (ranking), não como driver de perturbação dominante.

**`Vol_t`** é pouco usada em H=3 mas **lidera a saliência em H=12** — coerente com o alvo *delta* e a inércia anual do estoque.

**Perfil temporal**: recência extrema em H=3 (razão t−0/t−11 ≈ 13×); memória distribuída em H=12 (≈ 3×).

**Seq-PDP (H=3, vencedor `C`)** das 4 features mais importantes — `qbal`, `pr`, `eto` e mês (`month_sin` convertido pela inversa do seno): mais `qbal` e `pr` recentes elevam Δ̂Vol. Leitura de **sensibilidade do modelo, não de efeito físico direto** (o PDP ignora interações, §13.2).

## § 12. Exportação de resultados

Todos os artefatos do artigo com prefixo `article_` (CSVs) ou na pasta
`figures/article/` — nada sobrescreve as saídas do `lstm_exploratory` (Partes I e II). Outputs são **gerados, nunca digitados**: re-rodar o
notebook regenera tudo.

In [ ]:
for h in HORIZONS:
    res[h]["by_fold"].to_csv(
        METRICS_DIR / f"article_results_by_fold_h{h}.csv", index=False)
    res[h]["train_val"].to_csv(
        METRICS_DIR / f"article_train_val_by_fold_h{h}.csv", index=False)
    res[h]["train_info"].to_csv(
        METRICS_DIR / f"article_train_info_by_fold_h{h}.csv", index=False)
    perf[h].to_csv(METRICS_DIR / f"article_performance_h{h}.csv")
    resumo_folds[h].to_csv(METRICS_DIR / f"article_rmse_folds_summary_h{h}.csv")
    rankings[h].to_csv(METRICS_DIR / f"article_ranking_h{h}.csv")
    res[h]["wf_all"].to_csv(
        PRED_DIR / f"article_preds_wf_h{h}.csv", index=False)
    wf_qr[h].to_csv(
        PRED_DIR / f"article_preds_qr_best_h{h}.csv", index=False)

    # Mapas xAI (L x p) — auditáveis sem re-rodar o notebook.
    L = XAI[h]["sal"].shape[0]
    idx = [f"t-{L - 1 - i}" for i in range(L)]
    pd.DataFrame(XAI[h]["sal"], index=idx, columns=XAI[h]["features"]).to_csv(
        METRICS_DIR / f"article_xai_saliency_h{h}.csv")
    pd.DataFrame(XAI[h]["pert"], index=idx, columns=XAI[h]["features"]).to_csv(
        METRICS_DIR / f"article_xai_perturbation_h{h}.csv")

qr_summary.to_csv(METRICS_DIR / "article_qr_summary.csv")
qr_reliability.to_csv(METRICS_DIR / "article_qr_reliability.csv")
qr_probplot.to_csv(METRICS_DIR / "article_qr_probplot.csv", index=False)
qr_triggers.to_csv(METRICS_DIR / "article_qr_triggers.csv")
dm_table.to_csv(METRICS_DIR / "article_diebold_mariano.csv", index=False)

# Remove checkpoints intermediários (artefatos de depuração dos loops).
for fpath in METRICS_DIR.glob("_checkpoint_article_*.csv"):
    fpath.unlink()

print("Exportação concluída:")
print(f"  predições -> {PRED_DIR}  (article_preds_*.csv)")
print(f"  métricas  -> {METRICS_DIR}  (article_*.csv)")
print(f"  figuras   -> {FIG_DIR}")

## § 13. Revisão crítica, limitações e referências

### 13.1. Decisões de protocolo (herdadas e auditadas)

* **Walk-forward, não split único** (Tashman, 2000; Bergmeir &
  Benítez, 2012): em versões anteriores do braço H=12 o split fixo era
  treino 1984–2015 / **validação 2016–2018** / teste 2019–2025, e o
  modelo com a melhor validação era o **pior** no teste — a validação
  fixa era cega à mudança de regime. O mecanismo está nos dados: a
  janela 2016–2018 cai inteiramente sobre a **recuperação pós-crise de
  2014–2015**, um reenchimento quase monotônico (jan/2016 = 80 hm³ →
  pico de 664 hm³), com armazenamento médio **+510 hm³ acima da cauda
  do treino** (média 2014–2015 = −57 hm³ de volume útil). A parada
  antecipada era, portanto, calibrada sobre um estoque anomalamente
  alto *em relação ao que o modelo acabara de ver* e depois testada em
  2019–2025, que inclui a estiagem de 2021–2022 (−28% da média
  1984–2025). Nuance honesta: em termos **climatológicos** a janela
  não é úmida (−15% da média de longo prazo) — o que a torna
  enganosa é o **contraste com o histórico recente**, não o nível
  absoluto. O walk-forward retreina a cada fold, faz cada validação
  acompanhar o regime do seu próprio teste e mede também a
  variabilidade entre janelas de avaliação.
* **W = 12 de validação**: adotada por análise de sensibilidade
  (empata em RMSE com 36; única janela com POD > 0 para secas severas).
* **`max_date` = fim do teste − H**: o parâmetro filtra a data de
  emissão (bug histórico do `lstm_exploratory` (Parte II) §13.2, corrigido e herdado
  já correto aqui — `n_test = 12` é verificado por assert em cada fold).
* **Baselines obrigatórios** (`docs/methodological_rules.md` §3.5): persistência-H,
  climatologia mensal expansiva e média do treino, todos recalculados
  por fold.

### 13.2. Limitações

* **Poucos eventos de seca** no período de teste (Q20) — todas as
  métricas categóricas têm incerteza amostral alta; PODs têm
  granularidade de 1/n_eventos.
* **Não-determinismo de CPU**: treino não é bit-a-bit reprodutível
  entre execuções mesmo com seeds fixas (~±10 hm³ no RMSE agregado do
  H=12 entre execuções completas; menor com 10 seeds, não nulo). Não
  interpretar dígitos decimais.
* **Blocos anuais** no walk-forward (adaptação de custo para redes; o
  walk-forward de referência do XGBoost desliza mês a mês).
* **Seleção no teste** (§9): o ranking usa o teste — os vencedores são
  "melhores candidatos atuais", a confirmar em dados futuros (2026+).
* **xAI é interpretativo, não causal** (§11): mapas de
  saliência/perturbação medem sensibilidade do modelo; Seq-PDP herda as
  limitações do PDP (média marginal, ignora interações).

### 13.3. Referências

* **Hao, Z., Singh, V. P., & Xia, Y. (2018).** Seasonal Drought
  Prediction: Advances, Challenges, and Future Prospects. *Reviews of
  Geophysics*, 56(1), 108–141. doi:10.1002/2016RG000549.
* **Tashman, L. J. (2000).** Out-of-sample tests of forecasting
  accuracy: an analysis and review. *International Journal of
  Forecasting*, 16(4), 437–450. doi:10.1016/S0169-2070(00)00065-0 —
  avaliação com **origem rolante** (*rolling-origin*), fundamento do
  protocolo walk-forward da §5.
* **Bergmeir, C., & Benítez, J. M. (2012).** On the use of
  cross-validation for time series predictor evaluation. *Information
  Sciences*, 191, 192–213. doi:10.1016/j.ins.2011.12.028 — por que, em
  séries temporais, a validação em blocos que avançam no tempo é
  preferível ao *hold-out* único e à CV aleatória.
* **Gupta, H. V., Kling, H., Yilmaz, K. K., & Martinez, G. F. (2009).**
  Decomposition of the mean squared error and NSE performance criteria.
  *Journal of Hydrology*, 377(1–2), 80–91 — KGE e decomposição r/α/β.
* **Knoben, W. J. M., Freer, J. E., & Woods, R. A. (2019).** Inherent
  benchmark or not? Comparing Nash–Sutcliffe and Kling–Gupta efficiency
  scores. *HESS*, 23, 4323–4331 — interpretação do KGE.
* **Van Loon, A. F. (2015).** Hydrological drought explained. *WIREs
  Water*, 2(4), 359–392 — limiar variável (percentis 70–90%).
* **Wilks, D. S. (2019).** *Statistical Methods in the Atmospheric
  Sciences* (4ª ed.). Elsevier — verificação categórica (POD/FAR/CSI/
  HSS) e discriminação × calibração.
* **Koenker, R., & Bassett, G. (1978).** Regression Quantiles.
  *Econometrica*, 46(1), 33–50 — perda *pinball* (§10).
* **Simonyan, K., Vedaldi, A., & Zisserman, A. (2014).** Deep Inside
  Convolutional Networks: Visualising Image Classification Models and
  Saliency Maps. *ICLR Workshop* — mapas de saliência por gradiente.
* **Friedman, J. H. (2001).** Greedy Function Approximation: A Gradient
  Boosting Machine. *Annals of Statistics*, 29(5) — Partial Dependence.
* **Fisher, A., Rudin, C., & Dominici, F. (2019).** All Models are
  Wrong, but Many are Useful. *JMLR*, 20(177) — importância por
  permutação/perturbação, base conceitual da sensibilidade da §11.